In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import torch
from src.data import PoseDataset, prepare_pose_graph_batch

ann_path = "data/crowdpose/annotations/crowdpose_val.json"
img_root = "data/crowdpose/images"
dataset = PoseDataset.from_path(ann_path, img_root)
dataloader = dataset.create_dataloader(batch_size=4, shuffle=False)

ex_batch = next(iter(dataloader))
prepare_pose_graph_batch(ex_batch)
print("Batch keys:", list(ex_batch.keys()))
print("node_pos shape:", ex_batch["node_pos"].shape)   # (B*N*K, 2)
print("edge_index shape:", ex_batch["edge_index"].shape)


Batch keys: ['image_id', 'img_path', 'keypoints', 'visibility', 'edge_index', 'node_pos', 'batch', 'batch_idx', 'pose_idx', 'kpt_type_idx', 'node_idx', 'canonical_visibility', 'root_pos', 'scale']
node_pos shape: torch.Size([60, 2])
edge_index shape: torch.Size([2, 136])


In [7]:
ex_batch["keypoints"].shape

torch.Size([4, 1, 14, 2])

In [3]:
# --- Diffusion model: forward (add noise) + backward (denoise) ---
from src.model import KeypointDiffusionModel, KeypointDiffusionConfig

device = "cuda" if torch.cuda.is_available() else "cpu"

config = KeypointDiffusionConfig(dim_hidden=128, num_layers=4)
model = KeypointDiffusionModel(config, num_keypoints=15).to(device)

node_pos     = ex_batch["node_pos"].to(device)          # (B*N*K, 2)
edge_index   = ex_batch["edge_index"].to(device)        # (2, B*N*E)
kpt_type_idx = ex_batch["kpt_type_idx"].reshape(-1).to(device)   # (B*N*K,)
batch_idx    = ex_batch["batch"].to(device)             # (B*N*K,)
B            = ex_batch["keypoints"].shape[0]

# Forward pass: add noise at a random timestep
t = torch.randint(1, config.num_train_steps + 1, (B,), device=device)
noisy_pos, noise = model.add_noise(node_pos, t, batch_idx)
print("noisy_pos:", noisy_pos.shape)

# Backward pass: predict noise
pred_noise = model.backward_process(
    noisy_node_pos=noisy_pos,
    edge_index=edge_index,
    kpt_type_idx=kpt_type_idx,
    batch_idx=batch_idx,
    timestep=t,
)
loss = torch.nn.functional.mse_loss(pred_noise, noise)
print(f"Noise prediction loss (random init): {loss.item():.4f}")

# DDIM sampling from pure noise
N = ex_batch["keypoints"].shape[1]
K = 15
num_nodes = B * N * K
sampled = model.ddim_sample(
    edge_index=edge_index,
    kpt_type_idx=kpt_type_idx,
    batch_idx=batch_idx,
    num_nodes=num_nodes,
    num_inference_steps=20,
    device=device,
)
print("DDIM sample shape:", sampled.shape)  # (B*N*K, 2)


noisy_pos: torch.Size([60, 2])
Noise prediction loss (random init): 1.0670
DDIM sample shape: torch.Size([60, 2])


In [ ]:
# --- DinoConditioner: per-keypoint local patch embeddings ---
import torchvision
from src.model import DinoConditioner

pretrained_model_name = "facebook/dinov3-vits16-pretrain-lvd1689m"

dim_image_cond = 128
conditioner = DinoConditioner(
    model_name=pretrained_model_name,
    out_dim=dim_image_cond,
    patch_size=16,
    image_size=224,
    neighborhood_radius=1,   # 3×3 = 9 patches per keypoint
).to(device)

# Load a PIL image so we can inspect the raw backbone outputs
from PIL import Image
ex_path = ex_batch["img_path"][0]
pil_img = Image.open(ex_path).convert("RGB")
img_h, img_w = pil_img.height, pil_img.width
print(f"Image size: {img_w}×{img_h}")

# ---- Raw backbone patch features (for inspection) ----
patch_feats = conditioner._extract_patch_features([pil_img])
print("Patch feature grid shape:", patch_feats.shape)   # (1, G*G, hidden_dim)
G = conditioner.grid_size
print(f"Grid: {G}×{G} = {G*G} patches  |  hidden_dim={patch_feats.shape[-1]}")

# ---- Per-keypoint local embeddings for the whole batch ----
# keypoints are (B, N, 14, 2) pixel-space from the dataloader
B_b, N_b = ex_batch["keypoints"].shape[:2]
pil_images = [Image.open(p).convert("RGB") for p in ex_batch["img_path"]]
img_hw     = torch.tensor([(im.height, im.width) for im in pil_images],
                          dtype=torch.float32, device=device)

kpt_emb = conditioner(
    pil_images=pil_images,
    kpt_pixel=ex_batch["keypoints"].to(device),   # (B, N, 14, 2)
    img_hw=img_hw,
)
print(f"\nPer-keypoint DINO embedding shape: {kpt_emb.shape}")   # (B, N, 14, dim_image_cond)
print("  B =", B_b, " N =", N_b, " K=14  dim =", dim_image_cond)

# The root embedding is the mean of its 3 anchor joints
from src.data import CROWDPOSE_JOINT_NAMES
j = CROWDPOSE_JOINT_NAMES
anchor_idx = [j["left_hip"], j["right_hip"], j["neck"]]
root_emb = kpt_emb[:, :, anchor_idx, :].mean(dim=2, keepdim=True)   # (B, N, 1, dim)
full_emb = torch.cat([root_emb, kpt_emb], dim=2)                    # (B, N, 15, dim)
print(f"Full (root+14 joints) embedding shape: {full_emb.shape}")    # (B, N, 15, dim_image_cond)


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Image shape: torch.Size([3, 835, 550])
DINOv3 pooled embedding shape: torch.Size([1, 384])
DINOv3 patch features shape: torch.Size([1, 200, 384])


In [ ]:
# --- PoseDiffusionModule.prepare_batch: graph + DINO conditioning in one call ---
from src.model import KeypointDiffusionConfig, KeypointDiffusionModel, PoseDiffusionModule

config_cond = KeypointDiffusionConfig(
    dim_hidden=128,
    num_layers=4,
    use_image_cond=True,
    dim_image_cond=128,
)
model_cond = KeypointDiffusionModel(config_cond, num_keypoints=15).to(device)

module = PoseDiffusionModule(
    model=model_cond,
    dino_model_name=pretrained_model_name,
    dino_neighborhood_radius=1,
).to(device)

# Reload a fresh batch so the demo is self-contained
import copy
batch = copy.deepcopy(ex_batch)

module.prepare_batch(batch)

print("Keys after prepare_batch:", [k for k in batch if k not in ex_batch])
print("node_pos shape   :", batch["node_pos"].shape)        # (B*N*15, 2)
print("image_cond shape :", batch["image_cond"].shape)      # (B*N*15, dim_image_cond)
print("edge_index shape :", batch["edge_index"].shape)      # (2, B*N*E)

# Quick sanity: image_cond attaches one embedding per node — shapes must match
assert batch["node_pos"].shape[0] == batch["image_cond"].shape[0], "Node count mismatch!"
print("\nAll shapes consistent ✓")

# Forward pass through the conditioned model
B_b = batch["keypoints"].shape[0]
t = torch.randint(1, config_cond.num_train_steps + 1, (B_b,), device=device)
noisy_pos, noise = model_cond.add_noise(batch["node_pos"].to(device), t, batch["batch"].to(device))

pred_noise = model_cond.backward_process(
    noisy_node_pos=noisy_pos,
    edge_index=batch["edge_index"].to(device),
    kpt_type_idx=batch["kpt_type_idx"].reshape(-1).to(device),
    batch_idx=batch["batch"].to(device),
    timestep=t,
    image_cond=batch["image_cond"],   # ← DINO conditioning injected here
)
print("pred_noise shape :", pred_noise.shape)    # (B*N*15, 2)
loss = torch.nn.functional.mse_loss(pred_noise, noise)
print(f"Loss (random init): {loss.item():.4f}")
